In [1]:
import re
from pathlib import Path
import shutil

In [7]:
import re

def est_ligne_parasite(ligne):
    ligne = ligne.strip()

    if not ligne:
        return False

    # Numéro de page isolé
    if re.match(r'^\d+$', ligne):
        return True

    # Titre courant répété du roman (ex. CHÉRIE / CHERIE)
    if re.match(r'^CH[ÉE]RIE$', ligne, flags=re.IGNORECASE):
        return True

    # Tables / listes éditoriales
    if re.match(
        r'^(table des matières(?: du titre)?|liste(?: générale)? des titres|table ?)$',
        ligne,
        flags=re.IGNORECASE
    ):
        return True

    # Chapitre / section / partie + chiffre romain
    if re.match(r'^chapitre\s+[IVXLCDM]+\s*$', ligne, flags=re.IGNORECASE):
        return True

    if re.match(r'^(section|sous-section|partie)\s+[IVXLCDM]+\s*$', ligne, flags=re.IGNORECASE):
        return True

    # Chiffre romain isolé
    if re.match(r'^[IVXLCDM]+\.?$', ligne, flags=re.IGNORECASE):
        return True

    # Titres de chapitre du type :
    # I. Le jardin du baobab
    # I – Le festin
    # I - Le festin
    # I Le festin
    if re.match(r'^[IVXLCDM]+(?:\.\s*|\s*[–-]\s*|\s+).+$', ligne):
        return True

    # Premier épisode / Deuxième partie / etc.
    if re.match(
        r'^(premier|première|deuxième|second|seconde|troisième|quatrième|cinquième|sixième|septième|huitième|neuvième|dixième)\s+'
        r'(épisode|partie|section|livre|chapitre)\b.*$',
        ligne,
        flags=re.IGNORECASE
    ):
        return True

    # Titre en un seul mot majuscule
    if re.match(r'^[A-ZÀÂÄÇÉÈÊËÎÏÔÖÙÛÜŸÆŒ]+$', ligne):
        return True

    # Ligne entièrement en majuscules sur plusieurs mots
    if re.match(r"^[A-ZÀÂÄÇÉÈÊËÎÏÔÖÙÛÜŸÆŒ0-9' -]{5,}$", ligne):
        mots = ligne.split()
        if len(mots) >= 2:
            return True

    return False
    

def nettoyer_texte(texte):
    
    texte = re.sub(
        r'(?ms)^\d+\.\s+.*?(?=^\d+\.\s+|^[IVXLCDM]+\.?\s*$|^[IVXLCDM]+(?:\.\s*|\s*[–-]\s*|\s+).+$|\Z)','',texte)
    
    # 1. Suppression des lignes parasites AVANT compactage
    lignes = texte.splitlines()
    lignes_filtrees = [ligne for ligne in lignes if not est_ligne_parasite(ligne)]
    texte = "\n".join(lignes_filtrees)

    # 2. Normalisation typographique légère
    texte = texte.replace("«", "")
    texte = texte.replace("»", "")
    texte = texte.replace('"', "")
    texte = texte.replace("—", " ")
    texte = texte.replace("–", " ")
    texte = texte.replace("<<", " ")
    texte = texte.replace(">>", " ")
    texte = texte.replace("<", " ")
    texte = texte.replace(">", " ")

    # enlève les tirets en début de ligne
    texte = re.sub(r'(?m)^\s*-\s*', '', texte)

    # 3. Réparation d’artefacts simples
    # mots coupés en fin de ligne : arran-\ngements -> arrangements
    texte = re.sub(r'(\w)-\s*\n\s*(\w)', r'\1\2', texte)
    texte = re.sub(r'(?<=[A-Za-zÀ-ÖØ-öø-ÿÆŒæœ])\d+', '', texte)

    # supprime espace avant ponctuation
    texte = re.sub(r'\s+([.,;:?!])', r'\1', texte)

    # 4. Compactage APRÈS suppression des lignes parasites
    texte = re.sub(r'\n+', '\n', texte)
    texte = re.sub(r'[ \t]+', ' ', texte).strip()

    # remise en bloc unique
    texte = re.sub(r'\s+', ' ', texte).strip()

    # 5. Protection des points de suspension
    texte = texte.replace("...", "<ELLIPSIS>")
    texte = texte.replace("…", "<ELLIPSIS_CHAR>")

    # 6. Protection des abréviations
    abreviations = [
        "M.", "MM.", "Mr.", "Mme.", "Mmes.", "Mlle.",
        "Dr.", "Pr.", "St.", "Ste.", "Sr.", "Jr.",
        "etc.", "cf.", "chap.", "vol.", "t." "1.", "2.", "3.", "4.", 
        "5.", "6.", "7.", "8.", "9."
    ]

    for abbr in abreviations:
        texte = texte.replace(abbr, abbr.replace(".", "<ABBR_DOT>"))

    # 7. Découpage après fin de phrase
    texte = re.sub(r'([.!?])\s+', r'\1\n', texte)

    # 8. Restaurations
    texte = texte.replace("<ABBR_DOT>", ".")
    texte = texte.replace("<ELLIPSIS>", "...")
    texte = texte.replace("<ELLIPSIS_CHAR>", "…")

    # 9. Nettoyage final
    texte = re.sub(r'\n{2,}', '\n', texte)
    lignes = [ligne.strip() for ligne in texte.split('\n') if ligne.strip()]
    texte = '\n'.join(lignes)

    return texte

In [ ]:
dossier_source = Path("/Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/Gongourt_Brute")

for source in dossier_source.glob("*.txt"):
    if source.stem.endswith("_clean"):
        continue

    destination = source.with_name(source.stem + "_clean.txt")

    try:
        with open(source, "r", encoding="utf-8") as f:
            contenu_brut = f.read()

        contenu_propre = nettoyer_texte(contenu_brut)

        with open(destination, "w", encoding="utf-8") as f:
            f.write(contenu_propre)

        print(f"Nettoyage terminé : {destination}")

    except FileNotFoundError:
        print(f"Fichier introuvable : {source}")

    except Exception as e:
        print(f"Erreur avec {source.name} : {e}")

Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/Gongourt_Brute/Sœur Philomène - Edmond de Goncourt & Jules de Goncourt_clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/Gongourt_Brute/Renée Mauperin copie_clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/Gongourt_Brute/Charles Demailly - Edmond de Goncourt & Jules de Goncourt_clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/Gongourt_Brute/Madame Gervaisais copie_clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/Gongourt_Brute/Germinie Lacerteux copie_clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpus/Data/Naturaliste/Textes_Brutes/Gongourt_Brute/une ingénue en puissance d'un névropathe_clean.txt
Nettoyage terminé : /Users/morganrichard/NLP-Naturalist-Corpu